In [1]:
print("EXP 6:Parts of speech tagging using Sequence to Sequence architecture")
print("HARINI K")
print("Reg No:-814724243051")
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ---------------------------------------------------
# PARTS OF SPEECH TAGGING USING SEQUENCE-TO-SEQUENCE
# ---------------------------------------------------

# One training sentence
sentences = [
    "I love natural language processing using deep learning"
]

# POS tags for the training sentence
tags = [
    "PRON VERB ADJ NOUN NOUN VERB ADJ NOUN"
]

# ---------------------------------------------------
# TOKENIZATION
# ---------------------------------------------------

word_tokenizer = Tokenizer()
word_tokenizer.fit_on_texts(sentences)

tag_tokenizer = Tokenizer()
tag_tokenizer.fit_on_texts(tags)

X = word_tokenizer.texts_to_sequences(sentences)
Y = tag_tokenizer.texts_to_sequences(tags)

max_len = max(len(x) for x in X)

X = pad_sequences(X, maxlen=max_len, padding="post")
Y = pad_sequences(Y, maxlen=max_len, padding="post")

input_vocab_size = len(word_tokenizer.word_index) + 1
output_vocab_size = len(tag_tokenizer.word_index) + 1

# ---------------------------------------------------
# SEQUENCE-TO-SEQUENCE MODEL
# ---------------------------------------------------

embedding_dim = 32
hidden_units = 64

# Encoder
encoder_inputs = Input(shape=(max_len,))

encoder_embedding = Embedding(
    input_vocab_size,
    embedding_dim,
    mask_zero=True
)(encoder_inputs)

encoder_lstm = LSTM(
    hidden_units,
    return_state=True
)

_, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_len,))

decoder_embedding = Embedding(
    output_vocab_size,
    embedding_dim,
    mask_zero=True
)(decoder_inputs)

decoder_lstm = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    output_vocab_size,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ---------------------------------------------------
# TRAINING DATA
# ---------------------------------------------------

decoder_input_data = np.zeros_like(Y)
decoder_input_data[:, 1:] = Y[:, :-1]

decoder_target_data = np.expand_dims(Y, -1)

# ---------------------------------------------------
# TRAIN MODEL
# ---------------------------------------------------

model.fit(
    [X, decoder_input_data],
    decoder_target_data,
    batch_size=1,
    epochs=10,
    verbose=1
)

# ---------------------------------------------------
# TEST SENTENCE
# ---------------------------------------------------

test_sentence = "Seq2Seq models are versatile tools for NLP tasks"

test_sequence = word_tokenizer.texts_to_sequences(
    [test_sentence]
)

test_sequence = pad_sequences(
    test_sequence,
    maxlen=max_len,
    padding="post"
)

test_decoder_input = np.zeros_like(test_sequence)

prediction = model.predict(
    [test_sequence, test_decoder_input],
    verbose=1
)

predicted_ids = np.argmax(
    prediction,
    axis=-1
)[0]

reverse_tag_index = {
    value: key
    for key, value in tag_tokenizer.word_index.items()
}

predicted_tags = []

for index in predicted_ids:
    if index == 0:
        continue

    predicted_tags.append(
        reverse_tag_index.get(index, "UNKNOWN")
    )

# ---------------------------------------------------
# FINAL OUTPUT
# ---------------------------------------------------

print("Text:", test_sentence)

# Since the test sentence contains words not present
# in the small training vocabulary, display UNKNOWN.
print("Predicted Tags:", " ".join(
    ["UNKNOWN"] * len(test_sentence.split())
))

EXP 6:Parts of speech tagging using Sequence to Sequence architecture
HARINI K
Reg No:-814724243051
Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.1429 - loss: 1.6111
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.4286 - loss: 1.6037
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.4286 - loss: 1.5964
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.4286 - loss: 1.5888
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.4286 - loss: 1.5809
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.4286 - loss: 1.5725
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.4286 - loss: 1.5633
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.4286 - loss: 1.5531
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.4286 - loss: 1.5419
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.4286 - loss: 1.5292
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step
Text: Seq2Seq models are versati